---
layout: post
codemirror: true
title: Space Game - Team Bob
description: A walkthrough of three custom game levels built with the GameEngine framework, CS 111 requirement alignment, mechanic deep-dives, and a future development roadmap.
permalink: /space-blog
author: Team Bob
---

## Quick Navigation

| | |
|--|--|
| [📋 CS 111 Requirements Overview](#cs-111-requirements--at-a-glance) | [🗺️ Development Roadmap](#development-roadmap) |
| [🚀 Level 1: Space Level](#level-1-space-level) | [🌀 Level 2: Alien Maze](#level-2-alien-maze) |
| [👾 Level 3: Alien Chase](#level-3-alien-chase--survive) | [🎮 Play All Three](#playing-all-three-levels-in-sequence) |

## About These Levels

Three custom levels were built using the **GameEngine framework** and exported from **GameBuilder**. Each level introduces a new mechanic — from basic NPC interaction to invisible maze navigation to a chase-and-survive AI challenge.

| # | Level | Core Mechanic |
|---|-------|---------------|
| 1 | **Space Level** | NPC dialogue + level transition via `continue = false` |
| 2 | **Alien Maze** | Invisible barriers, red glow on collision, timed run |
| 3 | **Alien Chase** | Chase AI, survival countdown, rage speed, E-key restart |

This blog documents each level's design alongside its alignment to the **CS 111 course requirements**. Requirements already demonstrated are marked ✅. Requirements planned for a future sprint are marked 🔲 and listed in the [Development Roadmap](#development-roadmap) at the bottom.

---

## CS 111 Requirements — At a Glance

The table below maps every CS 111 skill to its current status across the three levels. Each skill is covered in detail inside the level sections below.

### Control Structures

| Skill | Project Evidence | Status |
|-------|-----------------|--------|
| **Iteration** | `setInterval` chase loop, countdown loop, `for...of` in player search | ✅ All 3 levels |
| **Conditionals** | Collision guards, null checks, cooldown checks | ✅ All 3 levels |
| **Nested Conditions** | Rage mode: paused → moved → stillness timer → rage speed | ✅ Level 3 |

### Data Types

| Skill | Project Evidence | Status |
|-------|-----------------|--------|
| **Numbers** | Coordinates, `SCALE_FACTOR`, `remaining`, speeds, timer values | ✅ All 3 levels |
| **Strings** | Dialogue text, sprite `src` paths, DOM `cssText`, element `id`s | ✅ All 3 levels |
| **Booleans** | `visible`, `paused`, `frozen`, `levelTransitionTriggered` | ✅ All 3 levels |
| **Arrays** | `this.classes`, `dialogues[]`, `sources[]` in player search | ✅ All 3 levels |
| **Objects (JSON)** | `playerData`, `npcData1`, `CHASE_CONFIG`, `INTERACTION_CONFIG` | ✅ All 3 levels |

### Operators

| Skill | Project Evidence | Status |
|-------|-----------------|--------|
| **Mathematical** | `Math.sqrt(dx*dx + dy*dy)`, normalised vector `dx/dist`, `offsetX += normX * speed` | ✅ Level 3 |
| **String Operations** | Template literals `` `translate(${x}px, ${y}px)` ``, path concatenation `path + "/images/..."` | ✅ All 3 levels |
| **Boolean Expressions** | `&&` cooldown guard, `!==` null check, `||` fallback in player search | ✅ All 3 levels |

## Quick Controls Reference

| Level | Move | Interact |
|-------|------|----------|
| Space Level | ← ↑ → ↓ Arrow Keys | Walk into Chill Guy NPC |
| Alien Maze | W A S D | Walk to R2, press **E** |
| Alien Chase | W A S D | Press **E** near alien to restart after being caught |

---

## Level 1: Space Level

> An introductory level set on an alien planet. Navigate past a set of visible barriers to reach Chill Guy, who will send you to the next level.

[↑ Back to top](#quick-navigation)

### How It Works

**NPC Interaction and Level Transition**

The `Npc` object overrides two callbacks the engine calls at specific moments:

- `reaction()` — fires every frame the player's bounding box overlaps the NPC hitbox. Displays a dialogue line.
- `interact()` — fires when the player presses **E** within range:

```javascript
interact: function() {
    if (this.dialogueSystem) { this.showRandomDialogue(); }
    // Setting continue = false signals GameControl to advance to the next level
    if (this.gameEnv && this.gameEnv.gameControl &&
            this.gameEnv.gameControl.currentLevel) {
        this.gameEnv.gameControl.currentLevel.continue = false;
    }
}
```

**Barrier Layout**

Five `Barrier` objects use absolute pixel coordinates. `visible: true` lets the player see and navigate around them. `fromOverlay: true` marks them as builder-managed.

```javascript
const dbarrier_1 = {
    id: 'dbarrier_1', x: 700, y: 100, width: 150, height: 20,
    visible: true,
    hitbox: { widthPercentage: 0.0, heightPercentage: 0.0 },
    fromOverlay: true
};
```

### CS 111 Skills — Level 1

---

#### 🔄 Control Structures

**Iteration**
The engine iterates `this.classes` — an array — every frame to update and draw each game object. Every barrier, the player, and the NPC are processed through the same loop:
```javascript
// Engine internally does something like:
this.classes.forEach(entry => entry.instance.update());
```

**Conditionals**
Guard clauses check that objects exist before accessing them. Without this, the level crashes if the engine hasn't set up `gameControl` yet:
```javascript
if (this.gameEnv && this.gameEnv.gameControl && this.gameEnv.gameControl.currentLevel) {
    this.gameEnv.gameControl.currentLevel.continue = false;
}
```

**Nested Conditions**
The NPC dialogue system checks two conditions layered inside each other — first whether a dialogue system is available, then calls the appropriate method:
```javascript
reaction: function() {
    if (this.dialogueSystem) {          // outer: does the system exist?
        this.showReactionDialogue();    // inner action
    } else {
        console.log(this.greeting);    // fallback
    }
}
```

---

#### 📦 Data Types

**Numbers**
Every game entity is positioned and sized with numbers. `SCALE_FACTOR: 5` tells the engine to render the player at 1/5 of the viewport height. `STEP_FACTOR: 1000` controls movement speed.
```javascript
SCALE_FACTOR: 5,       // number — render scale
STEP_FACTOR: 1000,     // number — movement speed
INIT_POSITION: { x: 100, y: 300 },  // numbers — spawn coordinates
```

**Strings**
Strings are used for asset paths, element IDs, and dialogue text. The `+` operator concatenates the base path with the image filename:
```javascript
src: path + "/images/gamebuilder/sprites/astro.png",  // string concatenation
id: 'playerData',                                      // string identifier
greeting: 'Hi, I am a chill guy.',                    // string dialogue
```

**Booleans**
`visible: true` tells the engine to draw the barrier on screen. `levelTransitionTriggered: false` is a flag that flips to `true` to prevent the level from advancing twice:
```javascript
visible: true,                         // boolean — show or hide barrier
levelTransitionTriggered: false,       // boolean — transition guard flag
```

**Arrays**
`this.classes` is the core array of the level — each entry tells the engine which class to instantiate and what data to pass to it. The `dialogues` array holds all NPC lines:
```javascript
this.classes = [
    { class: GameEnvBackground, data: bgData },
    { class: Player,            data: playerData },
    { class: Npc,               data: npcData1 },
    { class: Barrier,           data: dbarrier_1 },
    // ... 4 more barriers
];
dialogues: ['Chill guy line 1', 'Good luck!', 'Time to move on!']
```

**Objects (JSON)**
Every entity is configured as a JS object — a key-value structure identical to JSON. Nested objects like `INIT_POSITION` and `hitbox` hold grouped values:
```javascript
const playerData = {
    id: 'playerData',
    SCALE_FACTOR: 5,
    INIT_POSITION: { x: 100, y: 300 },        // nested object
    hitbox: { widthPercentage: 0, heightPercentage: 0 }, // nested object
    keypress: { up: 38, left: 37, down: 40, right: 39 } // nested object
};
```

---

#### ➕ Operators

**Mathematical**
The NPC's initial position uses `Math.min()` to clamp row indices to valid sprite sheet bounds:
```javascript
right: { row: Math.min(1, 4 - 1), start: 0, columns: 3 },  // Math.min — clamping
```

**String Operations**
All asset paths are built by concatenating the engine's base `path` string with a relative image path:
```javascript
src: path + "/images/gamify/chillguy.png",   // string + operator
src: path + "/images/gamebuilder/sprites/astro.png",
```

**Boolean Expressions**
The level transition guard uses `&&` (AND) to chain three existence checks — all must be true before touching `currentLevel`:
```javascript
if (this.gameEnv && this.gameEnv.gameControl && this.gameEnv.gameControl.currentLevel) {
    // Safe to access .currentLevel — all three objects confirmed to exist
}
```

### ▶ Play Level 1

In [ ]:
%%js

// GAME_RUNNER: Navigate the alien planet and interact with Chill Guy to advance to the next level. Use Arrow Keys to move.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelSpace3 from '/assets/js/adventureGame/GameLevelSpace3.js';
export const gameLevelClasses = [GameLevelSpace3];
export { GameControl };

---

## Level 2: Alien Maze

> The walls are invisible. Touch one and you restart from the beginning. Navigate to R2 to complete the level — your time is being tracked!

[↑ Back to top](#quick-navigation)

### How It Works

**Invisible Barriers with Red Glow on Collision**

All barriers use `visible: false`. When a collision fires, `onCollide` injects a glowing `<div>` at the barrier's position and resets the player to spawn:

```javascript
const makeBarrier = (id, x, y, w, h) => ({
    id, x, y, width: w, height: h, visible: false,
    onCollide: function () {
        _glowBarrier(this);           // red flash at barrier's location
        const player = GameLevel2._findPlayer(gameEnv);
        if (player) {
            player.x = player.data.INIT_POSITION.x;  // reset to spawn
            player.y = player.data.INIT_POSITION.y;
        }
        GameLevel2._showRestartFlash();
    }
});
```

**Run Timer**

When the player clicks **START MISSION**, `Date.now()` is stored. When R2 is reached, elapsed seconds are computed and shown on the victory screen:
```javascript
const elapsed = Math.floor((Date.now() - GameLevel2._startTime) / 1000);
```

**Maze Layout**
Positions use fractions (0.0–1.0) scaled to the viewport. The left wall has a gap between `y: 0.35` and `y: 0.55` — the only way in:
```javascript
const mazeLeftTop    = makeBarrier('maze_left_top',    0.20, 0.15, 0.02, 0.20);
const mazeLeftBottom = makeBarrier('maze_left_bottom', 0.20, 0.55, 0.02, 0.30);
// entrance gap: y 0.35 → 0.55
```

### CS 111 Skills — Level 2

---

#### 🔄 Control Structures

**Iteration**
`_findPlayer()` uses a `for...of` loop to search through every object store the engine might use. It also uses `Array.isArray()` to handle both array and object-map formats:
```javascript
for (const list of sources) {                      // for...of iteration
    const arr = Array.isArray(list) ? list : Object.values(list);
    const found = arr.find(o => o?.data?.id === 'playerData');
    if (found) return found;
}
```

**Conditionals**
After finding the player, a null-check condition ensures we don't crash trying to reset a player that wasn't found:
```javascript
if (player) {                          // conditional: only reset if found
    player.x = player.data.INIT_POSITION.x;
    player.y = player.data.INIT_POSITION.y;
}
```

**Nested Conditions**
The DOM glow helper only creates a new glow element if one doesn't already exist (preventing stacking), then checks canvas position before positioning the div:
```javascript
function _glowBarrier(b) {
    if (document.getElementById(glowId)) return;  // outer: already exists?
    const canvas = document.querySelector('canvas');
    const cr = canvas ? canvas.getBoundingClientRect() : { left: 0, top: 0 }; // inner: canvas found?
    // ... create glow div
}
```

---

#### 📦 Data Types

**Numbers**
Fractional coordinates (0.0–1.0) scale automatically to any viewport. The timer computes elapsed time in whole seconds using `Math.floor()`:
```javascript
makeBarrier('maze_top', 0.20, 0.15, 0.60, 0.02);           // fractional numbers
const elapsed = Math.floor((Date.now() - _startTime) / 1000); // number arithmetic
```

**Strings**
Glow element IDs are constructed dynamically by concatenating a prefix string with the barrier's id string. DOM styles are set as CSS strings:
```javascript
const glowId = 'barrier-glow-' + barrierInstance.data.id;  // string concatenation
glow.style.cssText = `position: fixed; left: ${cr.left + bx}px; ...`; // template literal
```

**Booleans**
Every maze barrier has `visible: false` — a boolean that tells the engine not to draw it. `gameLevelTransitionTriggered` is a boolean flag preventing the level from advancing twice:
```javascript
visible: false,                         // boolean — barrier is hidden
gameLevelTransitionTriggered: false,    // boolean — transition guard
```

**Arrays**
The `sources` array in `_findPlayer()` lists every possible location where the engine might store game objects. `.filter(Boolean)` removes any `undefined` or `null` entries:
```javascript
const sources = [
    gameEnv?.gameObjects,
    gameEnv?.objects,
    gameEnv?.gameControl?.gameObjects,
].filter(Boolean);   // removes undefined/null — keeps only valid arrays
```

**Objects (JSON)**
The `makeBarrier` factory returns a plain JS object — a JSON-shaped data structure — with a method (`onCollide`) attached. This shows objects can hold both data and behaviour:
```javascript
const mazeTop = makeBarrier('maze_top', 0.20, 0.15, 0.60, 0.02);
// Returns: { id, x, y, width, height, visible, hitbox, onCollide: function(){...} }
```

---

#### ➕ Operators

**Mathematical**
The victory screen formats time with division and modulo to get minutes and seconds:
```javascript
const mins = Math.floor(elapsed / 60);      // division
const secs = String(elapsed % 60).padStart(2, '0'); // modulo
const timeStr = `${mins}:${secs}`;          // e.g. "1:07"
```

**String Operations**
Template literals build CSS position strings and element IDs at runtime:
```javascript
`left: ${cr.left + bx}px; top: ${cr.top + by}px;`   // template literal with math
'barrier-glow-' + barrierInstance.data.id             // concatenation
```

**Boolean Expressions**
Optional chaining (`?.`) is a short-circuit boolean operator — it returns `undefined` instead of crashing if the object doesn't exist:
```javascript
const found = arr.find(o => o?.data?.id === 'playerData' || o?.id === 'playerData');
//                          ^^ safe access    ^^ short-circuit OR
```

### ▶ Play Level 2

In [ ]:
%%js

// GAME_RUNNER: Find R2 hidden in the invisible maze — touch a wall and you restart! Use WASD to move.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevel2 from '/assets/js/adventureGame/GameLevel2.js';
export const gameLevelClasses = [GameLevel2];
export { GameControl };

---

## Level 3: Alien Chase — Survive!

> An alien slime is hunting you. Survive for 10 seconds to win. Stand still too long and it enters **rage mode**. If caught, press **E** near the alien to restart.

[↑ Back to top](#quick-navigation)

### How It Works

This level runs two independent systems alongside the standard game loop.

**`AlienChaseAI` — The Chase Loop**

Runs a `setInterval` at 33ms. Each tick it reads canvas positions with `getBoundingClientRect()`, computes a direction vector, and moves the alien with a CSS transform:
```javascript
npcCanvas.style.transform = `translate(${this.offsetX}px, ${this.offsetY}px)`;
```

**Stillness Detection → Rage Mode**

If the player hasn't moved more than 4px for 3 seconds, speed jumps from 1.2 to 3.8 px/tick and the HUD turns red:
```javascript
if ((now - this.stillSince) >= CHASE_CONFIG.STILL_THRESHOLD_MS) {
    this.currentSpeed = CHASE_CONFIG.RAGE_SPEED;
    this._setRageVisual(true);
}
```

**Full Interaction Chain**
```
Player enters Alien hitbox → NPC.reaction() → SurvivalManager.caught()
  frozen=true | countdown stopped | CAUGHT overlay shown | AI paused

Player presses E → NPC.interact() → SurvivalManager.reset()
  remaining=10s | frozen=false | overlays hidden | AI resumes to spawn

Countdown hits zero → SurvivalManager.showWin()
  WIN overlay | AI paused | after 4s → currentLevel.continue=false
```

### CS 111 Skills — Level 3

---

#### 🔄 Control Structures

**Iteration**
Two independent `setInterval` loops run concurrently — the chase AI at 33ms and the survival countdown at 100ms. Each has its own start, pause, and clear logic:
```javascript
// Chase AI loop — fires ~30 times per second
this.intervalId = setInterval(() => this._tick(), CHASE_CONFIG.TICK_MS);  // 33ms

// Survival countdown loop — fires 10 times per second
this.countdownId = setInterval(() => {
    this.remaining = Math.max(0, this.remaining - 100);
    this._updateHUD();
    if (this.remaining <= 0) { clearInterval(this.countdownId); this.showWin(); }
}, 100);
```

**Conditionals**
The catch cooldown guard uses a conditional to give the player a 1.2-second grace window after restarting, so the alien can't instantly re-catch them:
```javascript
caught() {
    if (this.frozen) return;     // already caught — do nothing
    if (this.lastResetTime !== null &&
        Date.now() - this.lastResetTime < INTERACTION_CONFIG.CATCH_COOLDOWN_MS) return;  // grace period
    this.frozen = true;
    // ...
}
```

**Nested Conditions**
The stillness detection inside `_tick()` nests conditions three levels deep:
```javascript
if (this.paused) return;                         // level 1: is AI paused?
if (this.lastPlayerPos !== null) {               // level 2: have we sampled a position yet?
    if (moved < CHASE_CONFIG.STILL_RADIUS) {     // level 3: did the player barely move?
        if (this.stillSince === null) this.stillSince = now; // level 4: start timer
        if ((now - this.stillSince) >= CHASE_CONFIG.STILL_THRESHOLD_MS) {
            this.currentSpeed = CHASE_CONFIG.RAGE_SPEED;     // trigger rage
        }
    }
}
```

---

#### 📦 Data Types

**Numbers**
Speed, distance, timer, and pixel offsets are all numbers. `Math.sqrt` computes the straight-line distance between the player and the alien each tick:
```javascript
const dist = Math.sqrt(dx * dx + dy * dy);  // Euclidean distance — number
this.offsetX += normX * this.currentSpeed;  // accumulated pixel offset — number
this.remaining = Math.max(0, this.remaining - 100); // countdown ms — number
```

**Strings**
The CSS transform is a template literal string built fresh every tick. The HUD timer display switches format in the last 3 seconds:
```javascript
npcCanvas.style.transform = `translate(${this.offsetX}px, ${this.offsetY}px)`;  // template literal
const display = secs <= 3 ? `⏱ ${secs.toFixed(1)}s` : `⏱ ${Math.ceil(secs)}s`; // conditional string
```

**Booleans**
Three boolean flags control the entire state machine of the level:
```javascript
AlienChaseAI.paused = false;      // boolean — is the chase loop frozen?
SurvivalManager.frozen = false;   // boolean — is the game in caught/win state?
visible: true;                    // boolean — is the barrier drawn on screen?
```

**Arrays**
`this.classes` again defines the level's game objects. The `dialogues` array holds three taunting lines, rotated randomly on E-key press:
```javascript
dialogues: [
    "Hah! Got you! Press E to restart.",
    "Every second near me costs you score!",
    "Can you escape in time?"
]
```

**Objects (JSON)**
`CHASE_CONFIG` and `INTERACTION_CONFIG` are plain JS objects used as centralised configuration stores — all tuning values in one place:
```javascript
const CHASE_CONFIG = {
    TICK_MS: 33, BASE_SPEED: 1.2, RAGE_SPEED: 3.8,
    STILL_THRESHOLD_MS: 3000, STILL_RADIUS: 4, STOP_RADIUS: 18,
    INIT_POSITION: { x: 500, y: 300 }   // nested object
};
```

---

#### ➕ Operators

**Mathematical**
The direction normalisation uses division and multiplication so the alien always moves at a consistent speed regardless of distance:
```javascript
const dist  = Math.sqrt(dx * dx + dy * dy);  // * and Math.sqrt
const normX = dx / dist;                      // division — normalise to unit vector
const normY = dy / dist;
this.offsetX += normX * this.currentSpeed;    // * — scale to current speed
this.offsetY += normY * this.currentSpeed;
```

**String Operations**
Every UI overlay and HUD text is built with template literals, embedding live numbers directly into strings:
```javascript
`translate(${this.offsetX}px, ${this.offsetY}px)`   // CSS transform string
`⏱ ${secs.toFixed(1)}s`                             // HUD timer string
```

**Boolean Expressions**
Compound boolean expressions gate every state transition. The cooldown check combines `!==` (not-equal), `&&` (AND), and `<` (less-than):
```javascript
if (this.lastResetTime !== null &&              // not-equal: has been reset before
    Date.now() - this.lastResetTime < INTERACTION_CONFIG.CATCH_COOLDOWN_MS) {  // less-than: within grace window
    return;  // all must be true to skip catching
}
```

### ▶ Play Level 3

In [ ]:
%%js

// GAME_RUNNER: Survive 10 seconds without being caught by the alien slime! Stand still and it goes into rage mode. Use WASD to move, E near the alien to restart.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelstuck_final from '/assets/js/adventureGame/GameLevelstuck_final.js';
export const gameLevelClasses = [GameLevelstuck_final];
export { GameControl };

---

## Playing All Three Levels in Sequence

Pass all three classes to `gameLevelClasses` and the engine advances automatically as each level's `continue` flag flips to `false`.

[↑ Back to top](#quick-navigation)

### ▶ Play All Three Levels

In [ ]:
%%js

// GAME_RUNNER: Play all three levels back to back — Space Level, Alien Maze, then Alien Chase. Arrow Keys for Level 1, WASD for Levels 2 and 3.

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameLevelSpace3 from '/assets/js/adventureGame/GameLevelSpace3.js';
import GameLevel2 from '/assets/js/adventureGame/GameLevel2.js';
import GameLevelstuck_final from '/assets/js/adventureGame/GameLevelstuck_final.js';
export const gameLevelClasses = [GameLevelSpace3, GameLevel2, GameLevelstuck_final];
export { GameControl };

---

## Development Roadmap

The features below are not yet built. Each is assigned to a future sprint with a concrete plan.

[↑ Back to top](#quick-navigation)

---

### 🔲 Sprint 1 — Backend Foundation & Auth
*Target: ~2 weeks*

| Feature | Plan |
|---------|------|
| **Collections / Lists** | Spring Boot `ScoreService` using `ArrayList<Score>` for leaderboard — demonstrates `add()`, `remove()`, iteration |
| **Sets** | `HashSet<String>` for unique player usernames — no duplicates without manual checks |
| **Dictionaries / Maps** | `HashMap<String, Integer>` for level high-score lookup: key = level name, value = best time |
| **Hashing** | BCrypt password hashing in Spring Security — store only hashed passwords, compare via `BCrypt.checkpw()` |
| **try/catch** | Wrap DOM overlay operations in JS `try/catch`; wrap all Java service layer DB calls |

---

### 🔲 Sprint 2 — Algorithms, Sorting & Testing
*Target: ~4 weeks*

| Feature | Plan |
|---------|------|
| **Sorting** | Leaderboard endpoint sorts `List<Score>` by time via `Comparator.comparingInt(Score::getTime)` |
| **Big-O Analysis** | Blog section: `_findPlayer()` = O(n), sort = O(n log n), HashMap lookup = O(1) |
| **Stacks / Queues** | Undo-last-move in maze using `Deque<Position>` — each step pushes `{x,y}`, pressing U pops it |
| **Input Validation** | Score submission form with JS validation before POST to leaderboard API |
| **Testing** | JUnit tests for `ScoreService` and `PlayerRepository`. Target >50% service layer coverage |
| **.then/.catch** | `fetch('/api/scores').then(res => res.json()).then(renderBoard).catch(showError)` |

---

### 🔲 Sprint 3 — Database, Trees & Async Assets
*Target: ~6 weeks*

| Feature | Plan |
|---------|------|
| **Database** | JPA/Hibernate `Score` entity with `@ManyToOne` to `Player`. Persist to SQLite (dev) / MySQL (prod) |
| **REST API** | `GET /api/scores`, `POST /api/scores`, `DELETE /api/scores/{id}` with `ResponseEntity` and proper HTTP codes |
| **Trees** | Decision Tree (Smile ML) to classify player skill from maze time + wall-hit count |
| **Async Loading** | `Promise.all([...assets.map(loadImage)])` — preload sprites before canvas appears |

---

### 🔲 Sprint 4 — Deployment & Graphs
*Target: ~8 weeks*

| Feature | Plan |
|---------|------|
| **Graphs** | Leaderboard challenge graph — BFS from new player to top rank via mutual beat-chains |
| **Docker** | `Dockerfile` + `docker-compose.yml` for Jekyll + Spring Boot + MySQL |
| **DNS** | Custom subdomain A record; verify with `dig` and `nslookup` |
| **nginx** | Reverse proxy: static files on port 80, `/api/*` → Spring Boot on port 8080 |
| **CI/CD** | GitHub Actions: JUnit on every PR, auto-deploy on merge to `main` |

---

## Image Asset Reference

| Asset | Path |
|-------|------|
| Alien planet background | `images/gamebuilder/bg/alien_planet.jpg` |
| Astronaut player sprite | `images/gamebuilder/sprites/astro.png` |
| Chill Guy NPC (Level 1) | `images/gamify/chillguy.png` |
| R2 robot NPC (Level 2) | `images/gamify/r2_idle.png` |
| Alien slime NPC (Level 3) | `images/gamebuilder/sprites/slime.png` |